# 12-02 TabPFN Top10 Model Selection, Frank-&-Hall-Validierung und Threshold-Kalibrierung

Dieses Notebook trifft alle Entscheidungen, die vor dem finalen Test feststehen müssen. Zuerst wird wie bisher die normale TabPFN-Konfiguration auf `Top10` und dem Validierungsjahr 2023 ausgewählt. Danach werden mit genau dieser Konfiguration drei binäre Validierungskanäle berechnet: `Top5`, `Top10` und `Top20`.

Aus diesen drei kumulativen Wahrscheinlichkeiten entsteht analog zum EBM-Notebook der Frank-&-Hall-Ensemble-Score:

```python
score_topk_raw_sum_valid = p_top5_valid + p_top10_valid + p_top20_valid
```

Der optimale Schwellenwert für die binäre Frage `actual_top10` wird ausschließlich auf der Validierung 2023 gesucht. `12-03` darf diesen Threshold später nur noch anwenden, nicht neu optimieren.


## Methodische Langfassung: Warum `12-02` die einzige Hyperparameter- und Threshold-Validierung enthält

`12-02` ist der methodische Sicherungspunkt zwischen Setup und echtem Zukunftstest. Alles, was eine Modellentscheidung ist, muss hier auf `X_valid` und damit auf Saison 2023 passieren. Der Testzeitraum 2024/2025 darf keine Parameter, keine Thresholds und keine Modellvariante mehr auswählen.

**Normale TabPFN-Parameter:** Die Parameterwahl bleibt bewusst einfach und stabil. Das Grid testet nur normale Classifier-Parameter wie `n_estimators`, `softmax_temperature`, `average_before_softmax` und `balance_probabilities`, sofern sie vom installierten `TabPFNClassifier` unterstützt werden. Thinking-Parameter gehören nicht in diese Validierung, weil sie eine andere Inferenzvariante darstellen und den Testvergleich verfälschen würden.

**Warum zuerst `Top10`:** `Top10` ist die zentrale sportliche Zielschwelle: Sie ist eng genug, um echte Spitzenleistung zu messen, aber weniger extrem selten als `Top5`. Deshalb wählt `Top10` die gemeinsame TabPFN-Konfiguration, die anschließend auf `Top5`, `Top10` und `Top20` übertragen wird. Diese gemeinsame Konfiguration macht die drei Kanäle vergleichbar und verhindert, dass jedes Target seine eigene, schwer interpretierbare Tuning-Geschichte bekommt.

**Frank-&-Hall-Logik:** Wie im EBM-Pfad werden drei kumulative binäre Fragen modelliert: Ist der Fahrer in den Top5, Top10 oder Top20? Diese Wahrscheinlichkeiten sind keine disjunkten Klassen. Ein Fahrer kann fachlich zugleich Top5, Top10 und Top20 sein. Deshalb werden die drei Rohwahrscheinlichkeiten addiert und als ordinaler Evidenzscore gelesen. Ein hoher Score bedeutet: Das Modell sieht gleichzeitig Signal für mehrere Top-K-Schwellen.

**Threshold-Kalibrierung:** Für die binäre Top10-Klassifikation reicht der Ranking-Score allein nicht aus. Deshalb wird auf 2023 ein F1-optimaler Threshold für `actual_top10` gesucht. Dieser Threshold ist ein validierter Entscheidungswert. Er darf im finalen Notebook 2024/2025 nur angewendet werden. Zusätzlich werden naive Baselines mit Threshold `0.5` berichtet: die Einzelkanäle bei `0.5` und das Ensemble bei `0.5`.

**Warum Konfusionsmatrizen wichtig sind:** Top-K-Targets sind stark unbalanciert. Eine hohe ROC-AUC kann gut aussehen, obwohl ein fester Threshold praktisch zu viele oder zu wenige Fahrer als Top10 markiert. Konfusionsmatrizen zeigen deshalb sichtbar, wie viele echte Top10-Fahrer getroffen werden und wie viele Fahrer fälschlich positiv werden. Precision, Recall und F1 ergänzen ROC-AUC und Average Precision um diese konkrete Schwellenwertsicht.

**Limitation:** Die drei TabPFN-Modelle werden separat trainiert. Es gibt keine Garantie, dass `P(Top5) <= P(Top10) <= P(Top20)` gilt. Diese Nicht-Monotonie wird nicht durch künstliche Differenzen kaschiert. Für die finale Rankinglogik bleibt die rohe Summe der drei Top-K-Wahrscheinlichkeiten der verbindliche Score.


## Metrik- und Statuslegende für die Validierung

Die erste Hälfte des Notebooks erzeugt die klassische `Top10`-Entscheidungstabelle:

- `roc_auc`: primäre Auswahlmetrik für die normale TabPFN-Konfiguration.
- `average_precision`: Zusatzmetrik für das unbalancierte Top10-Target.
- `runtime_status`: dokumentiert, ob ein Lauf aus Cache, API oder gar nicht verfügbar ist.
- `runtime_complete_for_comparison`: zeigt, ob echte API-Laufzeiten für faire Runtime-Vergleiche vorhanden sind.

Die zweite Hälfte erzeugt die Frank-&-Hall-Validierung:

- `p_top5_valid`, `p_top10_valid`, `p_top20_valid`: kumulative binäre TabPFN-Wahrscheinlichkeiten auf 2023.
- `score_topk_raw_sum_valid`: Summe der drei Wahrscheinlichkeiten, analog zur EBM-Logik.
- `threshold_source=native_0_5`: fester Schwellenwert `0.5`, ohne Optimierung.
- `threshold_source=optimized_valid_2023`: F1-optimaler Threshold auf `actual_top10`, ausschließlich aus 2023.
- `tabpfn_validation_confusion_matrices.csv`: 2x2-Konfusionsmatrizen für Einzelkanäle und Ensemble.
- `tabpfn_validation_classification_metrics.csv`: Precision, Recall, F1, ROC-AUC und Average Precision.


## Imports und Projektpfade

Der erste Codeblock richtet nur die Arbeitsumgebung ein. Die Projektwurzel wird direkt im Notebook gesucht, damit die Pfade auch dann stimmen, wenn das Notebook aus einem anderen Arbeitsverzeichnis gestartet wird.

**Schutz vor Unübersichtlichkeit:** Es wird keine Pfad-Hilfsfunktion angelegt. Die Suche ist als kurzer, sichtbarer Loop ausgeschrieben.

### Erwarteter Output

Die Pfadprüfung soll nicht nur zeigen, dass das Notebook läuft. Sie soll belegen, dass alle späteren Exporte in denselben Projektbaum schreiben. Besonders bei API-Caches ist das wichtig: Ein Cache-Hit aus einem falschen Ordner könnte sonst wie ein valider Modelllauf aussehen.

In [34]:
# Wir laden nur die Bibliotheken, die in diesem Notebook direkt gebraucht werden.
from itertools import product
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# Die Projektwurzel wird linear gesucht: vom aktuellen Ordner nach oben, bis Daten- und Notebook-Ordner gefunden werden.
start_path = Path.cwd()
PROJECT_ROOT = None
for candidate in [start_path, *start_path.parents]:
    has_model_data = (candidate / "data" / "model_data").exists()
    has_notebooks = (candidate / "src" / "Notebooks").exists()
    if has_model_data and has_notebooks:
        PROJECT_ROOT = candidate
        break

# Ohne Projektwurzel wären alle relativen Pfade unsicher; deshalb wird hier bewusst hart abgebrochen.
if PROJECT_ROOT is None:
    raise FileNotFoundError("Projektwurzel mit data/model_data und src/Notebooks nicht gefunden.")

MODEL_DATA_DIR = PROJECT_ROOT / "data" / "model_data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "data" / "models"
TABPFN_DIR = PROCESSED_DIR / "tabpfn"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"MODEL_DATA_DIR: {MODEL_DATA_DIR}")

path_overview = pd.DataFrame([
    {"name": "PROJECT_ROOT", "path": str(PROJECT_ROOT), "exists": PROJECT_ROOT.exists()},
    {"name": "MODEL_DATA_DIR", "path": str(MODEL_DATA_DIR), "exists": MODEL_DATA_DIR.exists()},
    {"name": "RESULT_BASE", "path": str(PROCESSED_DIR / "tabpfn"), "exists": (PROCESSED_DIR / "tabpfn").exists()},
    {"name": "MODEL_DIR", "path": str(MODEL_DIR), "exists": MODEL_DIR.exists()},
])

print("=" * 88)
print("TABPFN 12-02: Pfadprüfung und Arbeitskontext")
print("=" * 88)
display(path_overview)

PROJECT_ROOT: /Users/roberthendrich/GADA-Group3-Cycling-Stage-Prediction
MODEL_DATA_DIR: /Users/roberthendrich/GADA-Group3-Cycling-Stage-Prediction/data/model_data
TABPFN 12-02: Pfadprüfung und Arbeitskontext


,name,path,exists
0,PROJECT_ROOT,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
1,MODEL_DATA_DIR,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
2,RESULT_BASE,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
3,MODEL_DIR,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True


## Notebook-Konfiguration und feste Modelllogik

Dieser Abschnitt legt die unveränderlichen Regeln des Tuning-Notebooks fest. Dazu gehören Feature-Reihenfolge, Target-Definition, sichere API-Schalter, Output-Ordner und das geplante Parameter-Grid.

**Schutz vor Leakage:** Die Konfiguration benennt explizit `Top10` als einziges Tuning-Target. Testdaten werden in diesem Notebook nicht geladen.

**Limitation:** `RUN_API = False` bedeutet, dass ohne vorhandene Prediction-Caches keine neuen Modellmetriken entstehen. Das ist gewollt, weil API-Läufe bewusst aktiviert werden müssen.

### Warum die Konfiguration ausführlich angezeigt wird

Für TabPFN sind die geplanten Parameter weniger klassisch interpretierbar, aber sie müssen genauso transparent dokumentiert werden. Das geplante Grid wird deshalb als Tabelle ausgegeben, bevor irgendein Lauf stattfindet.

**Schutzregel:** Thinking-Parameter sind in dieser Zelle explizit als verboten für die Validierung markiert. Sie können später interessant sein, würden hier aber die Hyperparameterwahl mit einem finalen Testvergleich vermischen.

In [35]:
# API-Läufe sind standardmäßig ausgeschaltet, damit das Notebook beim Öffnen keine externen Kosten erzeugt.
RUN_API = True  # False: keine API-Läufe; True: API-Läufe werden durchgeführt, wenn sie noch nicht existieren.
FORCE_RERUN = False
SEED = 42
TUNING_TARGET = "top10"
PRIMARY_METRIC = "roc_auc"

# Die aktuelle tabpfn_client-API akzeptiert serverseitig nur n_estimators bis 8.
TABPFN_CLIENT_N_ESTIMATORS_MAX = 8

# Diese 17 Features müssen exakt zur exportierten Modelldatenmatrix passen.
FEATURE_COLUMNS = [
    "distance",
    "vertical_meters",
    "stage_nr",
    "team_tier",
    "age_at_race",
    "rider_bmi",
    "wind_stability_index",
    "weather_temp_mean",
    "weather_temp_trend",
    "weather_rain_prob_mean",
    "weather_precipitation_mean",
    "weather_humidity_mean",
    "gradient_final_km",
    "lag_rider_points_season",
    "lag_rider_rank_season",
    "lag_race_competitiveness_median",
    "lag_team_power_index",
]

TARGET_CONFIGS = {
    "top5": {
        "label": "Top5",
        "train_file": "y_top5_train.pkl",
        "valid_file": "y_top5_valid.pkl",
        "score_col": "p_top5_valid",
        "actual_col": "actual_top5",
    },
    "top10": {
        "label": "Top10",
        "train_file": "y_top10_train.pkl",
        "valid_file": "y_top10_valid.pkl",
        "score_col": "p_top10_valid",
        "actual_col": "actual_top10",
    },
    "top20": {
        "label": "Top20",
        "train_file": "y_top20_train.pkl",
        "valid_file": "y_top20_valid.pkl",
        "score_col": "p_top20_valid",
        "actual_col": "actual_top20",
    },
}

EXPECTED_SHAPES = {
    "train": (169349, 17),
    "valid": (8897, 17),
}

# Thinking-Parameter werden in 12-02 nicht validiert; sie gehören erst in den finalen Testvergleich.
VALIDATION_FORBIDDEN_PARAMS = {"thinking_mode", "thinking_effort", "thinking_metric", "thinking_timeout_s"}

# Das Full Grid wird später nur auf tatsächlich unterstützte Parameter reduziert.
PLANNED_VALIDATION_GRID = {
    "n_estimators": [4, 8],
    "softmax_temperature": [0.6, 0.8, 1.0, 1.2, 1.4],
    "average_before_softmax": [True, False],
    "balance_probabilities": [True, False],
}

RESULT_DIR = TABPFN_DIR / "02_tuning_top10"
PREDICTION_DIR = RESULT_DIR / "predictions"
VALIDATION_PREDICTION_DIR = RESULT_DIR / "validation_predictions"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

print(f"RUN_API: {RUN_API}")
print(f"Tuning-Target: {TUNING_TARGET}")
print(f"Primäre Metrik: {PRIMARY_METRIC}")

config_summary = pd.DataFrame([
    {"setting": "RUN_API", "value": RUN_API, "interpretation": "neue API-Läufe nur bei bewusster Aktivierung"},
    {"setting": "FORCE_RERUN", "value": FORCE_RERUN, "interpretation": "bestehende Caches bleiben nutzbar"},
    {"setting": "TUNING_TARGET", "value": TUNING_TARGET, "interpretation": "einziger Zielkanal für die Parameterwahl"},
    {"setting": "PRIMARY_METRIC", "value": PRIMARY_METRIC, "interpretation": "primäre Sortiermetrik der Entscheidungstabelle"},
    {"setting": "Frank-Hall score", "value": "p_top5_valid + p_top10_valid + p_top20_valid", "interpretation": "Ensemble-Score für validierten Top10-Threshold"},
    {"setting": "n_estimators_limit", "value": TABPFN_CLIENT_N_ESTIMATORS_MAX, "interpretation": "tabpfn_client/API-Limit"},
    {"setting": "VALIDATION_PREDICTION_DIR", "value": str(VALIDATION_PREDICTION_DIR), "interpretation": "drei Target-Predictions für 2023"},
])

grid_plan = pd.DataFrame([
    {"parameter": parameter, "planned_values": values, "validation_role": "testen, falls vom Classifier unterstützt"}
    for parameter, values in PLANNED_VALIDATION_GRID.items()
])
for forbidden in sorted(VALIDATION_FORBIDDEN_PARAMS):
    grid_plan.loc[len(grid_plan)] = [forbidden, [], "verboten in 12-02; nur finaler Testvergleich"]

print("=" * 88)
print("TABPFN 12-02: Notebook-Konfiguration")
print("=" * 88)
display(config_summary)
display(grid_plan)


RUN_API: True
Tuning-Target: top10
Primäre Metrik: roc_auc
TABPFN 12-02: Notebook-Konfiguration


,setting,value,interpretation
0,RUN_API,True,neue API-Läufe nur bei bewusster Aktivierung
1,FORCE_RERUN,False,bestehende Caches bleiben nutzbar
2,TUNING_TARGET,top10,einziger Zielkanal für die Parameterwahl
3,PRIMARY_METRIC,roc_auc,primäre Sortiermetrik der Entscheidungstabelle
4,Frank-Hall score,p_top5_valid + p_top10_valid + p_top20_valid,Ensemble-Score für validierten Top10-Threshold
5,n_estimators_limit,8,tabpfn_client/API-Limit
6,VALIDATION_PREDICTION_DIR,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,drei Target-Predictions für 2023


,parameter,planned_values,validation_role
0,n_estimators,"[4, 8]","testen, falls vom Classifier unterstützt"
1,softmax_temperature,"[0.6, 0.8, 1.0, 1.2, 1.4]","testen, falls vom Classifier unterstützt"
2,average_before_softmax,"[True, False]","testen, falls vom Classifier unterstützt"
3,balance_probabilities,"[True, False]","testen, falls vom Classifier unterstützt"
4,thinking_effort,[],verboten in 12-02; nur finaler Testvergleich
5,thinking_metric,[],verboten in 12-02; nur finaler Testvergleich
6,thinking_mode,[],verboten in 12-02; nur finaler Testvergleich
7,thinking_timeout_s,[],verboten in 12-02; nur finaler Testvergleich


## TabPFN-Client und unterstützte Parameter prüfen

Hier wird geprüft, welche Parameter der aktuell installierte `TabPFNClassifier` kennt. Die Prüfung steht vor dem Grid-Aufbau, damit nicht unterstützte Parameter gar nicht erst als echte Kandidaten behandelt werden.

**Konkretes Vorgehen:** Der Import versucht zuerst `tabpfn_client` und danach das lokale `tabpfn`-Paket. Wenn kein Classifier verfügbar ist, kann das Notebook trotzdem Caches lesen, aber keine neuen API-Läufe starten.

### Interpretation der Parameterprüfung

Die Tabelle trennt geplante Parameter von tatsächlich unterstützten Parametern. Das ist methodisch wichtig, weil sich die Client-Version oder API-Grenzen ändern können. Ein nicht unterstützter Parameter wird nicht still ignoriert, sondern in der Tabelle sichtbar als nicht testbar ausgewiesen.

Wenn alle vier normalen Parameter unterstützt werden, entsteht das vollständige 40er-Grid. Wenn weniger Parameter unterstützt werden, bleibt die Validierung trotzdem reproduzierbar, weil die tatsächlich getestete Kandidatentabelle exportiert wird.

In [36]:
# Der Classifier wird direkt importiert; wenn das Paket fehlt, bleibt der Notebook-Pfad für Cache-Auswertungen nutzbar.
TabPFNClassifier = None
client_import_error = None
try:
    from tabpfn_client import TabPFNClassifier as ImportedTabPFNClassifier
    TabPFNClassifier = ImportedTabPFNClassifier
    client_source = "tabpfn_client"
except Exception as exc_client:
    try:
        from tabpfn import TabPFNClassifier as ImportedTabPFNClassifier
        TabPFNClassifier = ImportedTabPFNClassifier
        client_source = "tabpfn"
    except Exception as exc_local:
        client_import_error = f"tabpfn_client: {exc_client}; tabpfn: {exc_local}"
        client_source = None

if TabPFNClassifier is None:
    warnings.warn(f"Kein TabPFNClassifier importierbar. API-Läufe bleiben deaktiviert. {client_import_error}")
    supported_params = {}
else:
    # get_params() ist die Quelle dafür, welche normalen Classifier-Parameter in das Grid dürfen.
    try:
        supported_params = TabPFNClassifier().get_params()
    except Exception as exc:
        warnings.warn(f"TabPFNClassifier konnte nicht instanziiert werden: {exc}")
        supported_params = {}

# Die Tabelle dokumentiert transparent, was getestet, ignoriert oder später erst final verglichen wird.
supported_rows = []
for parameter, planned_values in {**PLANNED_VALIDATION_GRID, **{parameter: [] for parameter in sorted(VALIDATION_FORBIDDEN_PARAMS)}}.items():
    supported_by_classifier = parameter in supported_params
    if parameter in VALIDATION_FORBIDDEN_PARAMS:
        validation_status = "nur finaler Test in 12-03"
    elif supported_by_classifier:
        validation_status = "test im Full Grid in 12-02"
    else:
        validation_status = "nicht unterstützt oder unbekannt"

    supported_rows.append({
        "parameter": parameter,
        "supported_by_classifier": supported_by_classifier,
        "planned_values": planned_values,
        "validation_status": validation_status,
        "client_source": client_source,
    })

supported_param_df = pd.DataFrame(supported_rows)
supported_param_df.to_csv(RESULT_DIR / "classifier_supported_params.csv", index=False)

support_counts = supported_param_df.groupby(["validation_status", "supported_by_classifier"], dropna=False).size().reset_index(name="count")

print("=" * 88)
print("TABPFN 12-02: Unterstützte Classifier-Parameter")
print("=" * 88)
display(supported_param_df)
print("Kompakte Statuszählung:")
display(support_counts)


TABPFN 12-02: Unterstützte Classifier-Parameter


,parameter,supported_by_classifier,planned_values,validation_status,client_source
0,n_estimators,True,"[4, 8]",test im Full Grid in 12-02,tabpfn_client
1,softmax_temperature,True,"[0.6, 0.8, 1.0, 1.2, 1.4]",test im Full Grid in 12-02,tabpfn_client
2,average_before_softmax,True,"[True, False]",test im Full Grid in 12-02,tabpfn_client
3,balance_probabilities,True,"[True, False]",test im Full Grid in 12-02,tabpfn_client
4,thinking_effort,True,[],nur finaler Test in 12-03,tabpfn_client
5,thinking_metric,True,[],nur finaler Test in 12-03,tabpfn_client
6,thinking_mode,True,[],nur finaler Test in 12-03,tabpfn_client
7,thinking_timeout_s,True,[],nur finaler Test in 12-03,tabpfn_client


Kompakte Statuszählung:


,validation_status,supported_by_classifier,count
0,nur finaler Test in 12-03,True,4
1,test im Full Grid in 12-02,True,4


## Top10-Daten laden und Split prüfen

Dieser Abschnitt lädt ausschließlich die Daten, die für die Top10-Validierung nötig sind. `X_test` und die Testtargets bleiben bewusst unberührt.

**Schutz vor Leakage:** Trainiert wird auf `X_train` mit Jahren bis 2022, validiert wird auf `X_valid` aus 2023. Die Assertions prüfen Feature-Reihenfolge, Zielwerte und Validierungsjahr.

### Interpretation der Datenprüfung

Die Datenprüfung ist die direkte Entsprechung zur Datenlade- und Target-Dokumentation im EBM-Notebook. Sie zeigt nicht nur Shapes, sondern auch die positive Rate des Validierungstargets. Diese Rate ist wichtig, weil `average_precision` bei unbalancierten Daten anders gelesen werden muss als `roc_auc`.

**Leakage-Schutz:** `X_test` taucht hier bewusst nicht auf. Wenn die Testdaten in diesem Notebook geladen würden, wäre die Trennung zwischen Auswahl und finaler Bewertung unnötig aufgeweicht.

In [37]:
# Features und die drei kumulativen Top-K-Targets werden direkt aus data/model_data geladen.
X_train = pd.read_pickle(MODEL_DATA_DIR / "X_train.pkl")
X_valid = pd.read_pickle(MODEL_DATA_DIR / "X_valid.pkl")

y_targets = {}
for target, cfg in TARGET_CONFIGS.items():
    y_targets[(target, "train")] = pd.read_pickle(MODEL_DATA_DIR / cfg["train_file"])
    y_targets[(target, "valid")] = pd.read_pickle(MODEL_DATA_DIR / cfg["valid_file"])

# Der bestehende Top10-Grid-Code nutzt diese sprechenden Aliase weiter.
y_top10_train = y_targets[("top10", "train")]
y_top10_valid = y_targets[("top10", "valid")]

# Gruppen, Metadaten und echte Ränge werden für stage-weite Prediction-Tabellen benötigt.
y_rank_valid = pd.read_pickle(MODEL_DATA_DIR / "y_rank_valid.pkl")
groups_valid = pd.read_pickle(MODEL_DATA_DIR / "groups_valid.pkl")
meta_valid = pd.read_pickle(MODEL_DATA_DIR / "meta_valid.pkl")

# Die Feature-Reihenfolge muss exakt mit dem Modellsetup übereinstimmen.
assert list(X_train.columns) == FEATURE_COLUMNS
assert list(X_valid.columns) == FEATURE_COLUMNS

# Shape-Abweichungen können durch erneuten Export entstehen; sie werden sichtbar gewarnt statt still ignoriert.
if X_train.shape != EXPECTED_SHAPES["train"]:
    warnings.warn(f"X_train Shape weicht ab: {X_train.shape}")
if X_valid.shape != EXPECTED_SHAPES["valid"]:
    warnings.warn(f"X_valid Shape weicht ab: {X_valid.shape}")

# Die Validierung muss ausschließlich aus 2023 stammen und alle Targets müssen binär codiert sein.
assert set(meta_valid["meta_year"].unique()) == {2023}
for target in TARGET_CONFIGS:
    assert pd.Series(y_targets[(target, "train")]).isin([0, 1]).all()
    assert pd.Series(y_targets[(target, "valid")]).isin([0, 1]).all()

print("X_train", X_train.shape)
print("X_valid", X_valid.shape)
print("Top10 train positive rate", float(pd.Series(y_top10_train).mean()))
print("Top10 valid positive rate", float(pd.Series(y_top10_valid).mean()))

split_check_df = pd.DataFrame([
    {"split": "train_selection", "rows": len(X_train), "features": X_train.shape[1], "years": "<=2022", "role": "Fit-Kontext für Top10-Validierung und Frank-&-Hall-Kanäle"},
    {"split": "validation", "rows": len(X_valid), "features": X_valid.shape[1], "years": ", ".join(map(str, sorted(meta_valid["meta_year"].unique()))), "role": "einzige Hyperparameter- und Threshold-Validierung"},
])

target_balance_rows = []
for target, cfg in TARGET_CONFIGS.items():
    for split_name in ["train", "valid"]:
        y_series = pd.Series(y_targets[(target, split_name)])
        target_balance_rows.append({
            "target": target,
            "label": cfg["label"],
            "split": "train_selection" if split_name == "train" else "validation",
            "rows": len(y_series),
            "positives": int(y_series.sum()),
            "positive_rate_percent": 100 * float(y_series.mean()),
        })
target_balance_df = pd.DataFrame(target_balance_rows)

top10_balance_df = target_balance_df[target_balance_df["target"] == "top10"].copy()

print("=" * 88)
print("TABPFN 12-02: Top-K-Daten und Klassenverteilung")
print("=" * 88)
display(split_check_df)
display(target_balance_df)


X_train (169349, 17)
X_valid (8897, 17)
Top10 train positive rate 0.05908508464768023
Top10 valid positive rate 0.06406653928290434
TABPFN 12-02: Top-K-Daten und Klassenverteilung


,split,rows,features,years,role
0,train_selection,169349,17,<=2022,Fit-Kontext für Top10-Validierung und Frank-&-...
1,validation,8897,17,2023,einzige Hyperparameter- und Threshold-Validierung


,target,label,split,rows,positives,positive_rate_percent
0,top5,Top5,train_selection,169349,5011,2.958978
1,top5,Top5,validation,8897,284,3.192087
2,top10,Top10,train_selection,169349,10006,5.908508
3,top10,Top10,validation,8897,570,6.406654
4,top20,Top20,train_selection,169349,19965,11.789264
5,top20,Top20,validation,8897,1138,12.790828


## Full Grid aufbauen

Das Grid wird als vollständiges kartesisches Produkt aufgebaut. Danach werden nur Parameter behalten, die der Classifier unterstützt und die nicht in 12-02 verboten sind.

**Interpretation:** Wenn alle vier normalen Parameter unterstützt werden, entstehen 40 Kandidaten. Wenn ein Parameter fehlt, wird er dokumentiert übersprungen, nicht stillschweigend verwendet.

### Interpretation des Full Grids

Das Full Grid verhindert eine sequenzielle Suche, bei der frühe Ergebnisse die später getesteten Parameterachsen beeinflussen. Alle unterstützten Kombinationen werden vorab festgelegt und danach gleich behandelt. Dadurch ist die Entscheidungstabelle am Ende einfacher zu interpretieren.

**Limitation:** Ein Full Grid kann teurer sein als ein schrittweiser Sweep. Deshalb werden Laufzeiten dokumentiert. Sie entscheiden aber nur bei praktisch gleich guter `roc_auc`, nicht als primäres Optimierungsziel.

In [38]:
# Das aktive Grid enthält nur unterstützte und erlaubte Validierungsparameter.
active_grid = {}
skipped_grid_rows = []

for parameter, planned_values in PLANNED_VALIDATION_GRID.items():
    planned_values = list(planned_values)

    # Thinking-Parameter wären methodisch falsch; diese Prüfung bleibt explizit sichtbar.
    if parameter in VALIDATION_FORBIDDEN_PARAMS:
        skipped_grid_rows.append({
            "parameter": parameter,
            "reason": "forbidden_in_validation",
            "planned_values": planned_values,
        })
        continue

    # Nicht unterstützte Parameter werden nicht an den Classifier oder die API weitergereicht.
    if parameter not in supported_params:
        skipped_grid_rows.append({
            "parameter": parameter,
            "reason": "not_supported_by_classifier",
            "planned_values": planned_values,
        })
        continue

    valid_values = []
    for value in planned_values:
        # Das tabpfn_client-Limit für n_estimators wird vor dem Lauf geprüft.
        if client_source == "tabpfn_client" and parameter == "n_estimators" and int(value) > TABPFN_CLIENT_N_ESTIMATORS_MAX:
            skipped_grid_rows.append({
                "parameter": parameter,
                "reason": "api_limit",
                "planned_values": [value],
                "error_message": f"n_estimators muss <= {TABPFN_CLIENT_N_ESTIMATORS_MAX} sein",
            })
            continue
        valid_values.append(value)

    if valid_values:
        active_grid[parameter] = valid_values
    else:
        skipped_grid_rows.append({
            "parameter": parameter,
            "reason": "no_valid_values_after_filtering",
            "planned_values": planned_values,
        })

# Aus dem aktiven Grid entsteht ein vollständiges Cartesian Product.
unique_candidate_runs = []
seen_param_json = set()
if active_grid:
    grid_keys = list(active_grid.keys())
    for values in product(*(active_grid[key] for key in grid_keys)):
        params = dict(zip(grid_keys, values))
        params = {key: value for key, value in params.items() if key in supported_params and key not in VALIDATION_FORBIDDEN_PARAMS}
        params_json = json.dumps(params, sort_keys=True, ensure_ascii=True, default=str)
        if params_json in seen_param_json:
            continue
        seen_param_json.add(params_json)
        run_label = f"grid_{len(unique_candidate_runs) + 1:03d}"
        unique_candidate_runs.append((run_label, params))
else:
    warnings.warn("Keine unterstützten Validierungsparameter für das Grid gefunden.")

# Die Kandidatentabelle ist der Nachweis, welche Konfigurationen tatsächlich verarbeitet werden.
candidate_grid_rows = []
for run_label, params in unique_candidate_runs:
    params_short = "defaults" if not params else "__".join(f"{key}-{value}" for key, value in sorted(params.items()))
    candidate_grid_rows.append({
        "run_label": run_label,
        "params_short": params_short,
        "params_json": json.dumps(params, sort_keys=True, ensure_ascii=True, default=str),
        **params,
    })

candidate_grid_df = pd.DataFrame(candidate_grid_rows)
skipped_grid_df = pd.DataFrame(skipped_grid_rows)

candidate_grid_df.to_csv(RESULT_DIR / "top10_tuning_candidate_grid.csv", index=False)
skipped_grid_df.to_csv(RESULT_DIR / "top10_tuning_grid_skipped_params.csv", index=False)

print(f"Grid-Runs: {len(unique_candidate_runs)}")

active_grid_df = pd.DataFrame([
    {"parameter": parameter, "active_values": values, "n_values": len(values)}
    for parameter, values in active_grid.items()
])

print("=" * 88)
print("TABPFN 12-02: Full-Grid-Kandidaten")
print("=" * 88)
display(active_grid_df)
display(candidate_grid_df.head(10))
if not skipped_grid_df.empty:
    print("Übersprungene Parameter oder Werte:")
    display(skipped_grid_df)

Grid-Runs: 40
TABPFN 12-02: Full-Grid-Kandidaten


,parameter,active_values,n_values
0,n_estimators,"[4, 8]",2
1,softmax_temperature,"[0.6, 0.8, 1.0, 1.2, 1.4]",5
2,average_before_softmax,"[True, False]",2
3,balance_probabilities,"[True, False]",2


,run_label,params_short,params_json,n_estimators,softmax_temperature,average_before_softmax,balance_probabilities
0,grid_001,average_before_softmax-True__balance_probabili...,"{""average_before_softmax"": true, ""balance_prob...",4,0.6,True,True
1,grid_002,average_before_softmax-True__balance_probabili...,"{""average_before_softmax"": true, ""balance_prob...",4,0.6,True,False
2,grid_003,average_before_softmax-False__balance_probabil...,"{""average_before_softmax"": false, ""balance_pro...",4,0.6,False,True
3,grid_004,average_before_softmax-False__balance_probabil...,"{""average_before_softmax"": false, ""balance_pro...",4,0.6,False,False
4,grid_005,average_before_softmax-True__balance_probabili...,"{""average_before_softmax"": true, ""balance_prob...",4,0.8,True,True
5,grid_006,average_before_softmax-True__balance_probabili...,"{""average_before_softmax"": true, ""balance_prob...",4,0.8,True,False
6,grid_007,average_before_softmax-False__balance_probabil...,"{""average_before_softmax"": false, ""balance_pro...",4,0.8,False,True
7,grid_008,average_before_softmax-False__balance_probabil...,"{""average_before_softmax"": false, ""balance_pro...",4,0.8,False,False
8,grid_009,average_before_softmax-True__balance_probabili...,"{""average_before_softmax"": true, ""balance_prob...",4,1.0,True,True
9,grid_010,average_before_softmax-True__balance_probabili...,"{""average_before_softmax"": true, ""balance_prob...",4,1.0,True,False


## Validierungsläufe ausführen oder aus Cache laden

Dieser Abschnitt ist der Kern des Notebooks. Jeder Grid-Kandidat wird nacheinander verarbeitet. Für jeden Lauf wird sichtbar entschieden: API-Limit, Cache-Hit, fehlender Cache bei `RUN_API=False` oder echter API-Lauf.

**Laufzeitregel:** Für Modellvergleiche zählt immer die ursprüngliche API-Zeit (`fit_seconds`, `predict_seconds`, `total_seconds`). `cache_load_seconds` beschreibt nur das lokale Lesen einer CSV und wird nicht als Modelllaufzeit interpretiert.

**Metrikregel:** Sobald Predictions vorliegen, werden `roc_auc` und `average_precision` auf `y_top10_valid` berechnet. Keine Testdaten, keine Rankingmetrik und kein Thinking-Parameter entscheidet in 12-02.

### Wie die Laufstatus zu lesen sind

Ein `api_run` bedeutet, dass Fit und Prediction neu ausgeführt wurden. Ein `cache_hit` bedeutet, dass vorhandene Predictions geladen wurden und die ursprüngliche API-Zeit aus Sidecar oder Runtime-CSV verfügbar war. `cache_hit_missing_api_runtime` ist inhaltlich verwertbar für Metriken, aber nicht für Laufzeitvergleiche. `skipped_missing_cache` bedeutet, dass bei `RUN_API=False` keine Prediction-Datei vorhanden war.

Diese Unterscheidung ist zentral, weil sonst die sehr schnelle lokale CSV-Ladezeit mit der echten TabPFN-Laufzeit verwechselt werden könnte.

In [39]:
# Leseschlüssel für diese lange Validierungszelle:
# 1. Jeder Grid-Kandidat wird einzeln und chronologisch verarbeitet.
# 2. Zuerst werden Parameter bereinigt und Dateipfade für Cache und Runtime-Sidecar festgelegt.
# 3. Danach entscheidet der Statuspfad: verbotener Parameter, API-Limit, Cache-Hit, fehlender Cache oder API-Lauf.
# 4. Bei Cache-Hits werden Metriken aus Predictions berechnet, aber Laufzeiten nur aus Sidecar/Runtime-CSV übernommen.
# 5. Bei API-Läufen werden Fit-Zeit, Prediction-Zeit und Gesamtzeit getrennt gemessen.
# 6. Erst wenn Predictions vorhanden sind, werden ROC-AUC und Average Precision berechnet.
# 7. Am Ende werden Runtime-, Metrik-, Run- und Usage-Tabellen getrennt exportiert.

from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

runtime_rows = []
metric_rows = []
run_rows = []
runtime_csv_path = RESULT_DIR / "top10_tuning_runtime.csv"

for run_label, candidate_params in unique_candidate_runs:
    print(f"Run: {run_label} {candidate_params}")

    # Die Parameter werden direkt im Loop final bereinigt: unterstützt, erlaubt und API-kompatibel.
    params = {key: value for key, value in candidate_params.items() if key in supported_params and key not in VALIDATION_FORBIDDEN_PARAMS}
    params_json = json.dumps(params, sort_keys=True, ensure_ascii=True, default=str)
    params_short = "defaults" if not params else "__".join(f"{key}-{value}" for key, value in sorted(params.items()))

    pred_path = PREDICTION_DIR / f"{run_label}.csv"
    sidecar_path = pred_path.with_name(f"{pred_path.stem}_runtime.json")
    run_start = perf_counter()

    # Runtime startet mit fehlenden API-Zeiten; sie wird später je nach Status gefüllt.
    runtime = {
        "run_label": run_label,
        "target": TUNING_TARGET,
        "params_json": params_json,
        "params_short": params_short,
        "rows_train": len(X_train),
        "rows_valid": len(X_valid),
        "fit_seconds": np.nan,
        "predict_seconds": np.nan,
        "total_seconds": np.nan,
        "cache_load_seconds": np.nan,
        "runtime_status": None,
        "runtime_time_source": "not_available",
        "api_run_timestamp": None,
        "runtime_complete_for_comparison": False,
        "error_message": None,
    }

    metrics = {
        "run_label": run_label,
        "target": TUNING_TARGET,
        "params_json": params_json,
        "params_short": params_short,
        "roc_auc": np.nan,
        "average_precision": np.nan,
        "prediction_path": str(pred_path),
        "status": None,
        "error_message": None,
        **params,
    }

    pred = None

    # Thinking-Parameter sind in der Validierung methodisch ausgeschlossen.
    forbidden_params = sorted(set(candidate_params) & VALIDATION_FORBIDDEN_PARAMS)
    if forbidden_params:
        runtime["runtime_status"] = "skipped_forbidden_params"
        runtime["error_message"] = f"Thinking-Parameter sind in 12-02 verboten: {forbidden_params}"
        runtime["total_seconds"] = round(perf_counter() - run_start, 3)
        metrics["status"] = runtime["runtime_status"]
        metrics["error_message"] = runtime["error_message"]

    # n_estimators > 8 wird für tabpfn_client gar nicht erst an die API geschickt.
    elif client_source == "tabpfn_client" and "n_estimators" in params and int(params["n_estimators"]) > TABPFN_CLIENT_N_ESTIMATORS_MAX:
        runtime["runtime_status"] = "skipped_api_limit"
        runtime["error_message"] = f"n_estimators muss <= {TABPFN_CLIENT_N_ESTIMATORS_MAX} sein"
        runtime["total_seconds"] = round(perf_counter() - run_start, 3)
        metrics["status"] = runtime["runtime_status"]
        metrics["error_message"] = runtime["error_message"]

    # Vorhandene Predictions werden zuerst gelesen; danach wird die ursprüngliche API-Zeit aus Sidecar oder Runtime-CSV rekonstruiert.
    elif pred_path.exists() and not FORCE_RERUN:
        cache_start = perf_counter()
        pred = pd.read_csv(pred_path)
        runtime["cache_load_seconds"] = round(perf_counter() - cache_start, 3)

        previous_runtime = None
        if sidecar_path.exists():
            with open(sidecar_path) as f:
                sidecar_record = json.load(f)
            has_fit = pd.notna(sidecar_record.get("fit_seconds"))
            has_predict = pd.notna(sidecar_record.get("predict_seconds"))
            has_total = pd.notna(sidecar_record.get("total_seconds"))
            if has_fit and has_predict and has_total:
                previous_runtime = sidecar_record
                previous_runtime["runtime_time_source"] = sidecar_record.get("runtime_time_source", "runtime_sidecar")

        if previous_runtime is None and runtime_csv_path.exists():
            previous_runtime_df = pd.read_csv(runtime_csv_path)
            if "run_label" in previous_runtime_df.columns:
                matches = previous_runtime_df[previous_runtime_df["run_label"].astype(str) == str(run_label)].copy()
                for _, previous_row in matches.iterrows():
                    previous_record = previous_row.to_dict()
                    has_fit = pd.notna(previous_record.get("fit_seconds"))
                    has_predict = pd.notna(previous_record.get("predict_seconds"))
                    has_total = pd.notna(previous_record.get("total_seconds"))
                    if has_fit and has_predict and has_total:
                        previous_runtime = previous_record
                        previous_runtime["runtime_time_source"] = "runtime_csv"
                        break

        if previous_runtime is None:
            warnings.warn(f"Cache-Hit für {run_label}, aber keine frühere API-Zeit gefunden. Laufzeitvergleich wird ausgeschlossen.")
            runtime["runtime_status"] = "cache_hit_missing_api_runtime"
            runtime["runtime_time_source"] = "missing_api_runtime"
        else:
            runtime["fit_seconds"] = previous_runtime.get("fit_seconds")
            runtime["predict_seconds"] = previous_runtime.get("predict_seconds")
            runtime["total_seconds"] = previous_runtime.get("total_seconds")
            runtime["runtime_status"] = "cache_hit"
            runtime["runtime_time_source"] = previous_runtime.get("runtime_time_source", "runtime_sidecar")
            runtime["api_run_timestamp"] = previous_runtime.get("api_run_timestamp")
            runtime["runtime_complete_for_comparison"] = True

            # Der Sidecar wird erneuert, damit zukünftige Cache-Hits die API-Zeiten direkt finden.
            sidecar_payload = {key: runtime.get(key) for key in [
                "run_label", "target", "fit_seconds", "predict_seconds", "total_seconds",
                "runtime_status", "runtime_time_source", "api_run_timestamp",
                "rows_train", "rows_valid", "params_json", "params_short",
            ]}
            with open(sidecar_path, "w") as f:
                json.dump(sidecar_payload, f, indent=2, ensure_ascii=True, default=str)

        metrics["status"] = runtime["runtime_status"]

    # Ohne Cache und ohne bewusst aktivierten API-Lauf wird der Kandidat sichtbar übersprungen.
    elif not RUN_API:
        runtime["runtime_status"] = "skipped_missing_cache"
        runtime["total_seconds"] = round(perf_counter() - run_start, 3)
        metrics["status"] = runtime["runtime_status"]

    # API-Lauf: Fit und Prediction werden getrennt gemessen, danach wird die positive Top10-Klasse extrahiert.
    else:
        if TabPFNClassifier is None:
            runtime["runtime_status"] = "missing_classifier"
            runtime["error_message"] = client_import_error
            runtime["total_seconds"] = round(perf_counter() - run_start, 3)
            metrics["status"] = runtime["runtime_status"]
            metrics["error_message"] = runtime["error_message"]
        else:
            try:
                api_timestamp = datetime.now(timezone.utc).isoformat()

                fit_start = perf_counter()
                model = TabPFNClassifier(**params)
                model.fit(X_train, y_top10_train)
                runtime["fit_seconds"] = round(perf_counter() - fit_start, 3)

                predict_start = perf_counter()
                proba = model.predict_proba(X_valid)
                classes = list(model.classes_)
                if 1 not in classes:
                    raise ValueError(f"Positive Klasse 1 fehlt in model.classes_: {classes}")
                p_top10_valid = proba[:, classes.index(1)]
                runtime["predict_seconds"] = round(perf_counter() - predict_start, 3)

                pred = meta_valid.reset_index(drop=True).copy()
                pred["stage_id"] = pd.Series(groups_valid).reset_index(drop=True).astype(str).values
                pred["actual_rank"] = pd.Series(y_rank_valid).reset_index(drop=True).values
                pred["actual_top10"] = pd.Series(y_top10_valid).reset_index(drop=True).astype(int).values
                pred["p_top10_valid"] = p_top10_valid
                pred["run_label"] = run_label
                pred["target"] = TUNING_TARGET
                for key, value in params.items():
                    pred[key] = value
                pred.to_csv(pred_path, index=False)

                runtime["total_seconds"] = round(perf_counter() - run_start, 3)
                runtime["runtime_status"] = "api_run"
                runtime["runtime_time_source"] = "current_api_run"
                runtime["api_run_timestamp"] = api_timestamp
                runtime["runtime_complete_for_comparison"] = True

                sidecar_payload = {key: runtime.get(key) for key in [
                    "run_label", "target", "fit_seconds", "predict_seconds", "total_seconds",
                    "runtime_status", "runtime_time_source", "api_run_timestamp",
                    "rows_train", "rows_valid", "params_json", "params_short",
                ]}
                with open(sidecar_path, "w") as f:
                    json.dump(sidecar_payload, f, indent=2, ensure_ascii=True, default=str)

                metrics["status"] = runtime["runtime_status"]
            except Exception as exc:
                runtime["runtime_status"] = "error"
                runtime["error_message"] = str(exc)
                runtime["total_seconds"] = round(perf_counter() - run_start, 3)
                metrics["status"] = runtime["runtime_status"]
                metrics["error_message"] = str(exc)

    # Wenn Predictions vorhanden sind, wird die Top10-Validierungsleistung berechnet.
    if pred is not None:
        y_true = pred["actual_top10"].astype(int)
        y_score = pred["p_top10_valid"].astype(float)
        if y_true.nunique() == 2:
            metrics["roc_auc"] = roc_auc_score(y_true, y_score)
            metrics["average_precision"] = average_precision_score(y_true, y_score)
        else:
            warnings.warn(f"{run_label}: ROC-AUC nicht berechenbar, weil nur eine Klasse in y_true vorkommt.")

    metric_rows.append({
        **metrics,
        "fit_seconds": runtime.get("fit_seconds"),
        "predict_seconds": runtime.get("predict_seconds"),
        "total_seconds": runtime.get("total_seconds"),
        "cache_load_seconds": runtime.get("cache_load_seconds"),
        "runtime_status": runtime.get("runtime_status"),
        "runtime_time_source": runtime.get("runtime_time_source"),
        "api_run_timestamp": runtime.get("api_run_timestamp"),
        "runtime_complete_for_comparison": runtime.get("runtime_complete_for_comparison"),
    })
    runtime_rows.append(runtime)
    run_rows.append({
        "run_label": run_label,
        "target": TUNING_TARGET,
        "params_short": params_short,
        "params_json": params_json,
        "status": metrics.get("status"),
    })

runtime_df = pd.DataFrame(runtime_rows)
metrics_df = pd.DataFrame(metric_rows)
runs_df = pd.DataFrame(run_rows)

runtime_df.to_csv(RESULT_DIR / "top10_tuning_runtime.csv", index=False)
metrics_df.to_csv(RESULT_DIR / "top10_tuning_metrics.csv", index=False)
runs_df.to_csv(RESULT_DIR / "top10_tuning_runs.csv", index=False)

# Usage wird angelegt, aber nicht mit Tokens oder API-Keys gefüllt.
usage_rows = []
for run_record in run_rows:
    usage_rows.append({
        "run_label": run_record["run_label"],
        "usage_status": "not_captured_in_notebook",
        "note": "Token/credit usage must be filled from client-specific usage APIs if available.",
    })
usage_df = pd.DataFrame(usage_rows)
usage_df.to_csv(RESULT_DIR / "top10_tuning_usage.csv", index=False)

if metrics_df.empty:
    status_summary = pd.DataFrame(columns=["runtime_status", "runs"])
    metric_preview = pd.DataFrame()
else:
    status_summary = metrics_df.groupby("runtime_status", dropna=False).size().reset_index(name="runs")
    preview_cols = [
        "run_label", "runtime_status", "roc_auc", "average_precision",
        "fit_seconds", "predict_seconds", "total_seconds", "cache_load_seconds",
        "runtime_time_source", "params_short",
    ]
    preview_cols = [col for col in preview_cols if col in metrics_df.columns]
    metric_preview = metrics_df.sort_values(["roc_auc", "average_precision"], ascending=[False, False], na_position="last")[preview_cols].head(12)

print("=" * 88)
print("TABPFN 12-02: Validierungsläufe, Status und Metrik-Preview")
print("=" * 88)
display(status_summary)
display(metric_preview)


Run: grid_001 {'n_estimators': 4, 'softmax_temperature': 0.6, 'average_before_softmax': True, 'balance_probabilities': True}
Run: grid_002 {'n_estimators': 4, 'softmax_temperature': 0.6, 'average_before_softmax': True, 'balance_probabilities': False}
Run: grid_003 {'n_estimators': 4, 'softmax_temperature': 0.6, 'average_before_softmax': False, 'balance_probabilities': True}
Run: grid_004 {'n_estimators': 4, 'softmax_temperature': 0.6, 'average_before_softmax': False, 'balance_probabilities': False}
Run: grid_005 {'n_estimators': 4, 'softmax_temperature': 0.8, 'average_before_softmax': True, 'balance_probabilities': True}
Run: grid_006 {'n_estimators': 4, 'softmax_temperature': 0.8, 'average_before_softmax': True, 'balance_probabilities': False}
Run: grid_007 {'n_estimators': 4, 'softmax_temperature': 0.8, 'average_before_softmax': False, 'balance_probabilities': True}
Run: grid_008 {'n_estimators': 4, 'softmax_temperature': 0.8, 'average_before_softmax': False, 'balance_probabilities':

,runtime_status,runs
0,cache_hit,40


,run_label,runtime_status,roc_auc,average_precision,fit_seconds,predict_seconds,total_seconds,cache_load_seconds,runtime_time_source,params_short
0,grid_001,cache_hit,0.783128,0.274043,1.509,24.145,25.687,0.016,current_api_run,average_before_softmax-True__balance_probabili...
1,grid_002,cache_hit,0.783128,0.274043,0.425,23.259,23.722,0.010,current_api_run,average_before_softmax-True__balance_probabili...
4,grid_005,cache_hit,0.783128,0.274043,0.423,23.092,23.553,0.010,current_api_run,average_before_softmax-True__balance_probabili...
5,grid_006,cache_hit,0.783128,0.274043,0.426,23.520,23.978,0.012,current_api_run,average_before_softmax-True__balance_probabili...
8,grid_009,cache_hit,0.783128,0.274043,0.428,23.694,24.163,0.010,current_api_run,average_before_softmax-True__balance_probabili...
9,grid_010,cache_hit,0.783128,0.274043,0.627,23.490,24.149,0.011,current_api_run,average_before_softmax-True__balance_probabili...
12,grid_013,cache_hit,0.783128,0.274043,0.631,22.903,23.567,0.010,current_api_run,average_before_softmax-True__balance_probabili...
13,grid_014,cache_hit,0.783128,0.274043,0.426,23.317,23.780,0.010,current_api_run,average_before_softmax-True__balance_probabili...
16,grid_017,cache_hit,0.783128,0.274043,0.425,23.104,23.568,0.010,current_api_run,average_before_softmax-True__balance_probabili...
17,grid_018,cache_hit,0.783128,0.274043,0.427,23.698,24.167,0.010,current_api_run,average_before_softmax-True__balance_probabili...


## Entscheidungstabelle und Export der besten Konfiguration

Die Entscheidungstabelle sortiert nach der dokumentierten Regel: zuerst `roc_auc`, dann `average_precision`, dann ursprüngliche API-Laufzeit und zuletzt einfachere Konfigurationen. Die beste Zeile wird als JSON gespeichert und in `12-03` wieder geladen.

**Interpretation:** Wenn keine gültige Metrik vorhanden ist, schreibt das Notebook trotzdem Tabellen mit Statusinformationen. Das ist besser als ein stiller Abbruch, weil sichtbar bleibt, ob Caches fehlen oder API-Läufe deaktiviert sind.

### Interpretation der Entscheidungstabelle

Die beste Zeile ist nicht automatisch die schnellste Zeile. Zuerst zählt `roc_auc`, dann `average_precision`. Erst danach werden Laufzeit und Einfachheit der Konfiguration als Tie-Breaker genutzt. Diese Reihenfolge entspricht der methodischen Logik: Zunächst muss das Modell Top10-Fahrer gut trennen; erst bei ähnlicher Leistung wird ein günstigerer Lauf bevorzugt.

In [40]:
# Nur Läufe mit echter ROC-AUC können für die Auswahl verwendet werden.
valid_metrics = metrics_df.dropna(subset=["roc_auc"]).copy() if not metrics_df.empty else pd.DataFrame()

if valid_metrics.empty:
    warnings.warn("Keine gültigen Metriken vorhanden. Entweder RUN_API=True setzen oder passende Prediction-Caches bereitstellen.")
    decision_table = metrics_df.copy()
else:
    # Einfachere Konfigurationen werden als letzter Tie-Breaker bevorzugt.
    if "n_estimators" in valid_metrics.columns:
        valid_metrics["n_estimators_sort"] = pd.to_numeric(valid_metrics["n_estimators"], errors="coerce")
    else:
        valid_metrics["n_estimators_sort"] = np.nan
    n_params_sort_values = []
    for params_json_value in valid_metrics["params_json"]:
        if pd.notna(params_json_value) and params_json_value:
            n_params_sort_values.append(len(json.loads(params_json_value)))
        else:
            n_params_sort_values.append(0)
    valid_metrics["n_params_sort"] = n_params_sort_values

    decision_table = valid_metrics.sort_values(
        ["roc_auc", "average_precision", "total_seconds", "n_estimators_sort", "n_params_sort"],
        ascending=[False, False, True, True, True],
        na_position="last",
    ).reset_index(drop=True)

decision_table.to_csv(RESULT_DIR / "top10_tuning_decision_table.csv", index=False)

if not valid_metrics.empty:
    best = decision_table.iloc[0].to_dict()
    best_params = json.loads(best["params_json"])
    selected = {
        "model_family": "TabPFNClassifier",
        "selection_target": "top10",
        "selection_metric": "roc_auc",
        "selection_train_context": "X_train, meta_year <= 2022",
        "selection_validation_context": "X_valid, meta_year == 2023",
        "selection_train_rows": int(len(X_train)),
        "validation_rows": int(len(X_valid)),
        "params": best_params,
        "excluded_from_validation": sorted(VALIDATION_FORBIDDEN_PARAMS),
        "validation_metrics": {
            "roc_auc": None if pd.isna(best.get("roc_auc")) else float(best.get("roc_auc")),
            "average_precision": None if pd.isna(best.get("average_precision")) else float(best.get("average_precision")),
        },
        "validation_runtime": {
            "fit_seconds": None if pd.isna(best.get("fit_seconds")) else float(best.get("fit_seconds")),
            "predict_seconds": None if pd.isna(best.get("predict_seconds")) else float(best.get("predict_seconds")),
            "total_seconds": None if pd.isna(best.get("total_seconds")) else float(best.get("total_seconds")),
            "runtime_time_source": best.get("runtime_time_source"),
            "api_run_timestamp": best.get("api_run_timestamp"),
        },
        "selection_reason": "Best validation roc_auc; average_precision, original API runtime and simpler configuration used as tie-breakers.",
    }

    with open(RESULT_DIR / "tabpfn_selected_classifier_params.json", "w") as f:
        json.dump(selected, f, indent=2, ensure_ascii=True)

    # Für den späteren Modellvergleich wird der beste Top10-Validierungslauf XGBoost-kompatibel abgelegt.
    best_pred_path = Path(best["prediction_path"])
    if best_pred_path.exists():
        best_pred = pd.read_csv(best_pred_path).copy()
        tabpfn_validation_results = best_pred.copy()
        tabpfn_validation_results["raw_prediction"] = tabpfn_validation_results["p_top10_valid"].astype(float)
        tabpfn_validation_results["true_rank"] = tabpfn_validation_results["actual_rank"]
        tabpfn_validation_results = tabpfn_validation_results.sort_values(
            ["stage_id", "raw_prediction"],
            ascending=[True, False],
        ).reset_index(drop=True)
        tabpfn_validation_results["predicted_rank"] = (
            tabpfn_validation_results.groupby("stage_id")["raw_prediction"].rank(method="first", ascending=False)
        )
        tabpfn_validation_results["forced_ordinal_class"] = np.select(
            [
                tabpfn_validation_results["predicted_rank"] <= 5,
                tabpfn_validation_results["predicted_rank"] <= 10,
                tabpfn_validation_results["predicted_rank"] <= 20,
            ],
            [3, 2, 1],
            default=0,
        )
        model_result_cols = [
            "meta_year", "meta_name", "meta_current_team", "meta_race", "stage_nr", "stage_id",
            "raw_prediction", "true_rank", "predicted_rank", "forced_ordinal_class",
        ]
        tabpfn_validation_results[model_result_cols].to_pickle(MODEL_DIR / "tabpfn_top10_validation_results.pkl")
    else:
        warnings.warn(f"Beste Prediction-Datei fehlt, data/models-Output wird nicht geschrieben: {best_pred_path}")
else:
    selected = None

if selected is None:
    selected_summary = pd.DataFrame([{
        "status": "no_selection",
        "reason": "Keine gültige ROC-AUC vorhanden; Caches fehlen oder RUN_API=False.",
    }])
else:
    selected_summary = pd.DataFrame([
        {"field": "model_family", "value": selected["model_family"]},
        {"field": "selection_target", "value": selected["selection_target"]},
        {"field": "selection_metric", "value": selected["selection_metric"]},
        {"field": "params", "value": json.dumps(selected["params"], ensure_ascii=True, sort_keys=True)},
        {"field": "roc_auc", "value": selected["validation_metrics"]["roc_auc"]},
        {"field": "average_precision", "value": selected["validation_metrics"]["average_precision"]},
        {"field": "total_seconds", "value": selected["validation_runtime"]["total_seconds"]},
    ])

print("=" * 88)
print("TABPFN 12-02: Entscheidungstabelle und ausgewählte Konfiguration")
print("=" * 88)
display(selected_summary)
display(decision_table.head(15))

TABPFN 12-02: Entscheidungstabelle und ausgewählte Konfiguration


,field,value
0,model_family,TabPFNClassifier
1,selection_target,top10
2,selection_metric,roc_auc
3,params,"{""average_before_softmax"": true, ""balance_prob..."
4,roc_auc,0.783128
5,average_precision,0.274043
6,total_seconds,23.553


,run_label,target,params_json,params_short,roc_auc,average_precision,prediction_path,status,error_message,n_estimators,softmax_temperature,average_before_softmax,balance_probabilities,fit_seconds,predict_seconds,total_seconds,cache_load_seconds,runtime_status,runtime_time_source,api_run_timestamp,runtime_complete_for_comparison,n_estimators_sort,n_params_sort
0,grid_005,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,0.783128,0.274043,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,cache_hit,None,4,0.8,True,True,0.423,23.092,23.553,0.010,cache_hit,current_api_run,2026-06-30T20:07:13.460860+00:00,True,4,4
1,grid_013,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,0.783128,0.274043,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,cache_hit,None,4,1.2,True,True,0.631,22.903,23.567,0.010,cache_hit,current_api_run,2026-06-30T20:10:25.815942+00:00,True,4,4
2,grid_017,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,0.783128,0.274043,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,cache_hit,None,4,1.4,True,True,0.425,23.104,23.568,0.010,cache_hit,current_api_run,2026-06-30T20:12:01.294069+00:00,True,4,4
3,grid_002,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,0.783128,0.274043,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,cache_hit,None,4,0.6,True,False,0.425,23.259,23.722,0.010,cache_hit,current_api_run,2026-06-30T20:06:02.252089+00:00,True,4,4
4,grid_014,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,0.783128,0.274043,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,cache_hit,None,4,1.2,True,False,0.426,23.317,23.780,0.010,cache_hit,current_api_run,2026-06-30T20:10:49.387060+00:00,True,4,4
5,grid_006,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,0.783128,0.274043,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,cache_hit,None,4,0.8,True,False,0.426,23.520,23.978,0.012,cache_hit,current_api_run,2026-06-30T20:07:37.018925+00:00,True,4,4
6,grid_010,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,0.783128,0.274043,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,cache_hit,None,4,1.0,True,False,0.627,23.490,24.149,0.011,cache_hit,current_api_run,2026-06-30T20:09:13.088926+00:00,True,4,4
7,grid_009,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,0.783128,0.274043,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,cache_hit,None,4,1.0,True,True,0.428,23.694,24.163,0.010,cache_hit,current_api_run,2026-06-30T20:08:48.920975+00:00,True,4,4
8,grid_018,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,0.783128,0.274043,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,cache_hit,None,4,1.4,True,False,0.427,23.698,24.167,0.010,cache_hit,current_api_run,2026-06-30T20:12:24.866090+00:00,True,4,4
9,grid_001,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,0.783128,0.274043,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,cache_hit,None,4,0.6,True,True,1.509,24.145,25.687,0.016,cache_hit,current_api_run,2026-06-30T20:05:36.557595+00:00,True,4,4


## Frank-&-Hall-Validierung und Threshold-Kalibrierung

Nach der normalen `Top10`-Parameterwahl wird die ausgewählte TabPFN-Konfiguration unverändert auf alle drei kumulativen Targets angewendet. Damit folgt der TabPFN-Pfad der gleichen Logik wie das EBM-Notebook: drei binäre Schwellenwertmodelle erzeugen drei Wahrscheinlichkeiten, und ihre Summe wird als ordinaler Ensemble-Score gelesen.

Wichtig ist die Rollenverteilung: `12-02` sucht den optimalen Threshold für `actual_top10`, weil hier die Validierung 2023 liegt. Der finale Test in `12-03` darf diesen Wert nur laden und anwenden. Dadurch bleibt der Testzeitraum frei von nachträglicher Kalibrierung.


In [41]:
# Leseschlüssel für die Frank-&-Hall-Zelle:
# 1. Die in der Top10-Validierung gewählte TabPFN-Konfiguration wird geladen oder aus der aktuellen Auswahl übernommen.
# 2. Mit genau diesen Parametern werden Top5-, Top10- und Top20-Predictions für 2023 erzeugt oder aus Cache gelesen.
# 3. Aus den drei kumulativen Wahrscheinlichkeiten entsteht score_topk_raw_sum_valid.
# 4. Native Thresholds bei 0.5 werden als Baseline dokumentiert.
# 5. Der F1-optimale Ensemble-Threshold für actual_top10 wird ausschließlich auf 2023 gesucht.
# 6. Alle Metriken und Konfusionsmatrizen werden als CSVs exportiert.

selected_path = RESULT_DIR / "tabpfn_selected_classifier_params.json"
if selected is None and selected_path.exists():
    with open(selected_path) as f:
        selected = json.load(f)
    warnings.warn("Keine frische Auswahl aus der aktuellen Notebook-Ausführung vorhanden. Lade vorhandene selected JSON für die Frank-&-Hall-Validierung.")

if selected is None:
    selected_params = {}
    warnings.warn("Keine ausgewählte TabPFN-Konfiguration verfügbar. Frank-&-Hall-Validierung kann erst nach gültiger Top10-Auswahl laufen.")
else:
    selected_params = selected.get("params", {}).copy()

selected_params = {key: value for key, value in selected_params.items() if key in supported_params and key not in VALIDATION_FORBIDDEN_PARAMS}
if client_source == "tabpfn_client" and "n_estimators" in selected_params and int(selected_params["n_estimators"]) > TABPFN_CLIENT_N_ESTIMATORS_MAX:
    warnings.warn(f"n_estimators={selected_params['n_estimators']} überschreitet das API-Limit; setze auf {TABPFN_CLIENT_N_ESTIMATORS_MAX}.")
    selected_params["n_estimators"] = TABPFN_CLIENT_N_ESTIMATORS_MAX

selected_params_json = json.dumps(selected_params, sort_keys=True, ensure_ascii=True, default=str)
validation_target_predictions = {}
validation_runtime_rows = []

for target, cfg in TARGET_CONFIGS.items():
    pred_path = VALIDATION_PREDICTION_DIR / f"tabpfn_validation_predictions_{target}_2023.csv"
    sidecar_path = pred_path.with_name(f"{pred_path.stem}_runtime.json")
    run_start = perf_counter()
    pred = None

    runtime = {
        "target": target,
        "params_json": selected_params_json,
        "prediction_path": str(pred_path),
        "fit_seconds": np.nan,
        "predict_seconds": np.nan,
        "total_seconds": np.nan,
        "cache_load_seconds": np.nan,
        "runtime_status": None,
        "runtime_time_source": "not_available",
        "api_run_timestamp": None,
        "runtime_complete_for_comparison": False,
        "error_message": None,
    }

    if pred_path.exists() and not FORCE_RERUN:
        cache_start = perf_counter()
        pred = pd.read_csv(pred_path)
        runtime["cache_load_seconds"] = round(perf_counter() - cache_start, 3)
        runtime["runtime_status"] = "cache_hit"
        if sidecar_path.exists():
            with open(sidecar_path) as f:
                sidecar_record = json.load(f)
            runtime["fit_seconds"] = sidecar_record.get("fit_seconds", np.nan)
            runtime["predict_seconds"] = sidecar_record.get("predict_seconds", np.nan)
            runtime["total_seconds"] = sidecar_record.get("total_seconds", np.nan)
            runtime["runtime_time_source"] = sidecar_record.get("runtime_time_source", "runtime_sidecar")
            runtime["api_run_timestamp"] = sidecar_record.get("api_run_timestamp")
            runtime["runtime_complete_for_comparison"] = bool(sidecar_record.get("runtime_complete_for_comparison", False))
        else:
            runtime["runtime_time_source"] = "cache_without_sidecar"

    elif target == "top10" and "best" in globals() and isinstance(best, dict) and best.get("prediction_path") and Path(best.get("prediction_path")).exists() and not FORCE_RERUN:
        cache_start = perf_counter()
        pred = pd.read_csv(Path(best["prediction_path"])).copy()
        pred = pred[[
            "meta_year", "meta_name", "meta_current_team", "meta_race", "stage_nr", "stage_id",
            "actual_rank", "actual_top10", "p_top10_valid",
        ]].copy()
        pred["target"] = target
        pred["params_json"] = selected_params_json
        pred.to_csv(pred_path, index=False)
        runtime["cache_load_seconds"] = round(perf_counter() - cache_start, 3)
        runtime["runtime_status"] = "cache_hit_from_selected_top10_grid"
        runtime["runtime_time_source"] = "selected_top10_grid_prediction"

    elif not RUN_API:
        runtime["runtime_status"] = "skipped_missing_cache"
        runtime["total_seconds"] = round(perf_counter() - run_start, 3)

    elif TabPFNClassifier is None:
        runtime["runtime_status"] = "missing_classifier"
        runtime["error_message"] = client_import_error
        runtime["total_seconds"] = round(perf_counter() - run_start, 3)

    elif selected is None:
        runtime["runtime_status"] = "missing_selected_params"
        runtime["error_message"] = "tabpfn_selected_classifier_params.json fehlt oder enthält keine gültige Auswahl."
        runtime["total_seconds"] = round(perf_counter() - run_start, 3)

    else:
        try:
            api_timestamp = datetime.now(timezone.utc).isoformat()
            fit_start = perf_counter()
            model = TabPFNClassifier(**selected_params)
            model.fit(X_train, y_targets[(target, "train")])
            runtime["fit_seconds"] = round(perf_counter() - fit_start, 3)

            predict_start = perf_counter()
            proba = model.predict_proba(X_valid)
            classes = list(model.classes_)
            if 1 not in classes:
                raise ValueError(f"Positive Klasse 1 fehlt in model.classes_: {classes}")
            positive_scores = proba[:, classes.index(1)]
            runtime["predict_seconds"] = round(perf_counter() - predict_start, 3)

            pred = meta_valid.reset_index(drop=True).copy()
            pred["stage_id"] = pd.Series(groups_valid).reset_index(drop=True).astype(str).values
            pred["actual_rank"] = pd.Series(y_rank_valid).reset_index(drop=True).values
            pred[cfg["actual_col"]] = pd.Series(y_targets[(target, "valid")]).reset_index(drop=True).astype(int).values
            pred[cfg["score_col"]] = positive_scores
            pred["target"] = target
            pred["params_json"] = selected_params_json
            pred.to_csv(pred_path, index=False)

            runtime["total_seconds"] = round(perf_counter() - run_start, 3)
            runtime["runtime_status"] = "api_run"
            runtime["runtime_time_source"] = "current_api_run"
            runtime["api_run_timestamp"] = api_timestamp
            runtime["runtime_complete_for_comparison"] = True

            sidecar_payload = {key: runtime.get(key) for key in [
                "target", "fit_seconds", "predict_seconds", "total_seconds", "runtime_status",
                "runtime_time_source", "api_run_timestamp", "runtime_complete_for_comparison",
                "params_json", "prediction_path",
            ]}
            with open(sidecar_path, "w") as f:
                json.dump(sidecar_payload, f, indent=2, ensure_ascii=True, default=str)
        except Exception as exc:
            runtime["runtime_status"] = "error"
            runtime["error_message"] = str(exc)
            runtime["total_seconds"] = round(perf_counter() - run_start, 3)

    validation_runtime_rows.append(runtime)
    if pred is not None:
        validation_target_predictions[target] = pred

validation_runtime_df = pd.DataFrame(validation_runtime_rows)
validation_runtime_df.to_csv(RESULT_DIR / "tabpfn_validation_predictions_runtime.csv", index=False)

if set(validation_target_predictions) == set(TARGET_CONFIGS):
    validation_combined = validation_target_predictions["top5"].copy()
    if "target" in validation_combined.columns:
        validation_combined = validation_combined.drop(columns=["target"])
    validation_combined["actual_top10"] = validation_target_predictions["top10"]["actual_top10"].reset_index(drop=True).astype(int)
    validation_combined["actual_top20"] = validation_target_predictions["top20"]["actual_top20"].reset_index(drop=True).astype(int)
    validation_combined["p_top10_valid"] = validation_target_predictions["top10"]["p_top10_valid"].reset_index(drop=True).astype(float)
    validation_combined["p_top20_valid"] = validation_target_predictions["top20"]["p_top20_valid"].reset_index(drop=True).astype(float)
    validation_combined["score_topk_raw_sum_valid"] = (
        validation_combined["p_top5_valid"].astype(float)
        + validation_combined["p_top10_valid"].astype(float)
        + validation_combined["p_top20_valid"].astype(float)
    )
    validation_combined.to_csv(RESULT_DIR / "tabpfn_validation_predictions_all_targets.csv", index=False)
else:
    validation_combined_columns = [
        "meta_year", "meta_name", "meta_current_team", "meta_race", "stage_nr", "stage_id",
        "actual_rank", "actual_top5", "actual_top10", "actual_top20",
        "p_top5_valid", "p_top10_valid", "p_top20_valid", "score_topk_raw_sum_valid",
    ]
    validation_combined = pd.DataFrame(columns=validation_combined_columns)
    validation_combined.to_csv(RESULT_DIR / "tabpfn_validation_predictions_all_targets.csv", index=False)
    missing_targets = sorted(set(TARGET_CONFIGS) - set(validation_target_predictions))
    warnings.warn(f"Frank-&-Hall-Validierung unvollständig. Fehlende Targets: {missing_targets}")

threshold_rows = []
classification_metric_rows = []
confusion_matrix_rows = []
best_threshold_record = None

if not validation_combined.empty:
    y_true_threshold = validation_combined["actual_top10"].astype(int)
    y_score_threshold = validation_combined["score_topk_raw_sum_valid"].astype(float)
    for threshold in np.round(np.arange(0.0, 3.0001, 0.01), 2):
        y_pred_threshold = (y_score_threshold >= threshold).astype(int)
        threshold_rows.append({
            "scope": "validation_2023",
            "target": "top10",
            "score_name": "score_topk_raw_sum_valid",
            "threshold": float(threshold),
            "precision": precision_score(y_true_threshold, y_pred_threshold, zero_division=0),
            "recall": recall_score(y_true_threshold, y_pred_threshold, zero_division=0),
            "f1": f1_score(y_true_threshold, y_pred_threshold, zero_division=0),
            "predicted_positives": int(y_pred_threshold.sum()),
            "actual_positives": int(y_true_threshold.sum()),
        })

    threshold_df = pd.DataFrame(threshold_rows)
    threshold_df = threshold_df.sort_values(
        ["f1", "precision", "threshold"],
        ascending=[False, False, False],
        na_position="last",
    ).reset_index(drop=True)
    best_threshold_record = threshold_df.iloc[0].to_dict()
    threshold_df["is_selected_threshold"] = False
    threshold_df.loc[0, "is_selected_threshold"] = True
else:
    threshold_df = pd.DataFrame(columns=[
        "scope", "target", "score_name", "threshold", "precision", "recall", "f1",
        "predicted_positives", "actual_positives", "is_selected_threshold",
    ])

threshold_df.to_csv(RESULT_DIR / "tabpfn_validation_thresholds.csv", index=False)

if not validation_combined.empty:
    evaluation_specs = []
    for target, cfg in TARGET_CONFIGS.items():
        evaluation_specs.append({
            "evaluation_name": f"{target}_native_0_5",
            "target": target,
            "actual_col": cfg["actual_col"],
            "score_col": cfg["score_col"],
            "score_name": cfg["score_col"],
            "threshold": 0.5,
            "threshold_source": "native_0_5",
        })
    evaluation_specs.append({
        "evaluation_name": "frank_hall_top10_native_0_5",
        "target": "top10",
        "actual_col": "actual_top10",
        "score_col": "score_topk_raw_sum_valid",
        "score_name": "score_topk_raw_sum_valid",
        "threshold": 0.5,
        "threshold_source": "native_0_5",
    })
    evaluation_specs.append({
        "evaluation_name": "frank_hall_top10_optimized_valid_2023",
        "target": "top10",
        "actual_col": "actual_top10",
        "score_col": "score_topk_raw_sum_valid",
        "score_name": "score_topk_raw_sum_valid",
        "threshold": float(best_threshold_record["threshold"]),
        "threshold_source": "optimized_valid_2023_f1",
    })

    for spec in evaluation_specs:
        y_true = validation_combined[spec["actual_col"]].astype(int)
        y_score = validation_combined[spec["score_col"]].astype(float)
        y_pred = (y_score >= float(spec["threshold"])).astype(int)
        matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
        roc_auc_value = roc_auc_score(y_true, y_score) if y_true.nunique() == 2 else np.nan
        average_precision_value = average_precision_score(y_true, y_score) if y_true.nunique() == 2 else np.nan

        classification_metric_rows.append({
            "scope": "validation_2023",
            "variant_name": "standard_selected",
            "evaluation_name": spec["evaluation_name"],
            "target": spec["target"],
            "actual_col": spec["actual_col"],
            "score_name": spec["score_name"],
            "threshold": float(spec["threshold"]),
            "threshold_source": spec["threshold_source"],
            "rows": int(len(y_true)),
            "actual_positives": int(y_true.sum()),
            "predicted_positives": int(y_pred.sum()),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "roc_auc": roc_auc_value,
            "average_precision": average_precision_value,
        })

        for actual_label, row_values in zip([0, 1], matrix):
            for predicted_label, count in zip([0, 1], row_values):
                confusion_matrix_rows.append({
                    "scope": "validation_2023",
                    "variant_name": "standard_selected",
                    "evaluation_name": spec["evaluation_name"],
                    "target": spec["target"],
                    "threshold": float(spec["threshold"]),
                    "threshold_source": spec["threshold_source"],
                    "actual_label": int(actual_label),
                    "predicted_label": int(predicted_label),
                    "count": int(count),
                })

classification_metric_columns = [
    "scope", "variant_name", "evaluation_name", "target", "actual_col", "score_name",
    "threshold", "threshold_source", "rows", "actual_positives", "predicted_positives",
    "precision", "recall", "f1", "roc_auc", "average_precision",
]
confusion_matrix_columns = [
    "scope", "variant_name", "evaluation_name", "target", "threshold", "threshold_source",
    "actual_label", "predicted_label", "count",
]
classification_metrics_df = pd.DataFrame(classification_metric_rows, columns=classification_metric_columns)
confusion_matrices_df = pd.DataFrame(confusion_matrix_rows, columns=confusion_matrix_columns)
classification_metrics_df.to_csv(RESULT_DIR / "tabpfn_validation_classification_metrics.csv", index=False)
confusion_matrices_df.to_csv(RESULT_DIR / "tabpfn_validation_confusion_matrices.csv", index=False)

if selected is not None and best_threshold_record is not None:
    selected["frank_hall_validation"] = {
        "score_name": "score_topk_raw_sum_valid",
        "threshold_target": "top10",
        "threshold_actual_col": "actual_top10",
        "threshold_metric": "f1",
        "threshold_validation_context": "X_valid, meta_year == 2023",
        "native_ensemble_threshold": 0.5,
        "optimized_threshold": float(best_threshold_record["threshold"]),
        "threshold_grid_min": 0.0,
        "threshold_grid_max": 3.0,
        "threshold_grid_step": 0.01,
        "optimized_threshold_metrics": {
            "precision": float(best_threshold_record["precision"]),
            "recall": float(best_threshold_record["recall"]),
            "f1": float(best_threshold_record["f1"]),
            "predicted_positives": int(best_threshold_record["predicted_positives"]),
            "actual_positives": int(best_threshold_record["actual_positives"]),
        },
        "component_scores": ["p_top5_valid", "p_top10_valid", "p_top20_valid"],
    }
    with open(selected_path, "w") as f:
        json.dump(selected, f, indent=2, ensure_ascii=True)

if validation_runtime_df.empty:
    validation_status_summary = pd.DataFrame(columns=["target", "runtime_status", "runs"])
else:
    validation_status_summary = validation_runtime_df.groupby(["target", "runtime_status"], dropna=False).size().reset_index(name="runs")

if classification_metrics_df.empty:
    metrics_preview = pd.DataFrame()
else:
    metrics_preview = classification_metrics_df.sort_values(["evaluation_name", "threshold_source"]).reset_index(drop=True)

print("=" * 88)
print("TABPFN 12-02: Frank-&-Hall-Validierung und Top10-Threshold")
print("=" * 88)
display(validation_status_summary)
if not threshold_df.empty:
    print("Ausgewählter Threshold auf Validierung 2023:")
    display(threshold_df[threshold_df["is_selected_threshold"] == True])
display(metrics_preview)
if not confusion_matrices_df.empty:
    print("Konfusionsmatrizen:")
    display(confusion_matrices_df.head(40))


00:05 Fitting... Done!
00:27 Predicting... Done!
00:04 Fitting... Done!
00:27 Predicting... Done!
TABPFN 12-02: Frank-&-Hall-Validierung und Top10-Threshold


,target,runtime_status,runs
0,top10,cache_hit,1
1,top20,api_run,1
2,top5,api_run,1


Ausgewählter Threshold auf Validierung 2023:


,scope,target,score_name,threshold,precision,recall,f1,predicted_positives,actual_positives,is_selected_threshold
0,validation_2023,top10,score_topk_raw_sum_valid,2.1,0.29878,0.34386,0.319739,656,570,True


,scope,variant_name,evaluation_name,target,actual_col,score_name,threshold,threshold_source,rows,actual_positives,predicted_positives,precision,recall,f1,roc_auc,average_precision
0,validation_2023,standard_selected,frank_hall_top10_native_0_5,top10,actual_top10,score_topk_raw_sum_valid,0.5,native_0_5,8897,570,4815,0.103011,0.870175,0.184215,0.783666,0.275056
1,validation_2023,standard_selected,frank_hall_top10_optimized_valid_2023,top10,actual_top10,score_topk_raw_sum_valid,2.1,optimized_valid_2023_f1,8897,570,656,0.298780,0.343860,0.319739,0.783666,0.275056
2,validation_2023,standard_selected,top10_native_0_5,top10,actual_top10,p_top10_valid,0.5,native_0_5,8897,570,1419,0.202255,0.503509,0.288587,0.783128,0.274043
3,validation_2023,standard_selected,top20_native_0_5,top20,actual_top20,p_top20_valid,0.5,native_0_5,8897,1138,1719,0.327516,0.494728,0.394120,0.759378,0.369215
4,validation_2023,standard_selected,top5_native_0_5,top5,actual_top5,p_top5_valid,0.5,native_0_5,8897,284,1163,0.116939,0.478873,0.187975,0.777716,0.193044


Konfusionsmatrizen:


,scope,variant_name,evaluation_name,target,threshold,threshold_source,actual_label,predicted_label,count
0,validation_2023,standard_selected,top5_native_0_5,top5,0.5,native_0_5,0,0,7586
1,validation_2023,standard_selected,top5_native_0_5,top5,0.5,native_0_5,0,1,1027
2,validation_2023,standard_selected,top5_native_0_5,top5,0.5,native_0_5,1,0,148
3,validation_2023,standard_selected,top5_native_0_5,top5,0.5,native_0_5,1,1,136
4,validation_2023,standard_selected,top10_native_0_5,top10,0.5,native_0_5,0,0,7195
5,validation_2023,standard_selected,top10_native_0_5,top10,0.5,native_0_5,0,1,1132
6,validation_2023,standard_selected,top10_native_0_5,top10,0.5,native_0_5,1,0,283
7,validation_2023,standard_selected,top10_native_0_5,top10,0.5,native_0_5,1,1,287
8,validation_2023,standard_selected,top20_native_0_5,top20,0.5,native_0_5,0,0,6603
9,validation_2023,standard_selected,top20_native_0_5,top20,0.5,native_0_5,0,1,1156


## Ergebnisinterpretation

Die wichtigste erste Tabelle dieses Notebooks ist `top10_tuning_decision_table.csv`. Sie dokumentiert die normale TabPFN-Konfiguration, fehlende Caches, übersprungene Kandidaten und Laufzeitquellen. Die wichtigste zweite Tabelle ist `tabpfn_validation_thresholds.csv`: Dort wird der F1-optimale Threshold für `actual_top10` ausschließlich aus der Validierung 2023 abgeleitet.

Für die Thesis ist entscheidend: `12-02` trifft sowohl die Modellparameterwahl als auch die Threshold-Kalibrierung. Erst `12-03` nutzt diese fixierten Entscheidungen für den echten Zukunftstest 2024/2025.

### Was aus diesem Notebook in `12-03` übernommen wird

Übernommen werden die normale Classifier-Konfiguration und der validierte Frank-&-Hall-Threshold aus `tabpfn_selected_classifier_params.json`. Nicht übernommen werden Testmetriken, Legacy-Rankings oder Thinking-Parameter. Dadurch bleibt der finale Test sauber: `12-03` lädt vorher festgelegte Entscheidungen, erzeugt 2024/2025-Predictions und bewertet sie ohne erneute Optimierung.

### Wie die Klassifikationsoutputs zu lesen sind

Die nativen Einzelkanäle mit Threshold `0.5` zeigen, wie `Top5`, `Top10` und `Top20` als separate binäre Fragen funktionieren. Das native Ensemble mit Threshold `0.5` ist nur eine naive Baseline, weil der Ensemble-Score von 0 bis 3 reicht. Der optimierte Ensemble-Threshold ist die eigentliche validierte Klassifikationsentscheidung für `actual_top10`.

Die Rankingbewertung bleibt davon getrennt. Ein Threshold sagt, ob ein Fahrer als Top10 klassifiziert wird. Das spätere Etappenranking sortiert dagegen alle Fahrer einer Etappe nach `score_topk_raw_sum`.
